In [5]:
from pathlib import Path
import sys
import torch


def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for c in candidates:
        if (c / "AAAI24_GARCH_NN_Reproduction").exists() and (c / "dataset").exists():
            return c
    raise RuntimeError("Cannot locate project root from current working directory")


def describe_gpu_and_pick_device():
    if not torch.cuda.is_available():
        print("CUDA available: False")
        return "cpu"

    device_idx = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(device_idx)
    total_gb = props.total_memory / (1024 ** 3)
    print("CUDA available: True")
    print(f"GPU: {props.name} (index={device_idx})")
    print(f"Total VRAM: {total_gb:.2f} GB")
    print(f"CUDA capability: {props.major}.{props.minor}")
    return f"cuda:{device_idx}"


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DEFAULT_DEVICE = describe_gpu_and_pick_device()

print(f"Project root: {PROJECT_ROOT}")
print(f"Python executable: {sys.executable}")
print(f"Selected device: {DEFAULT_DEVICE}")

CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU (index=0)
Total VRAM: 4.00 GB
CUDA capability: 8.6
Project root: D:\UIT\1003_EPA_PROJECT\1003_EPA-Project_UIT
Python executable: d:\UIT\1003_EPA_PROJECT\1003_EPA-Project_UIT\.venv\Scripts\python.exe
Selected device: cuda:0


In [6]:
DATASET_DIR = PROJECT_ROOT / "dataset"
dataset_files = sorted(DATASET_DIR.glob("*.csv"))

print(f"Dataset dir: {DATASET_DIR}")
print(f"Found {len(dataset_files)} CSV files")
for p in dataset_files:
    print(" -", p.name)

assert dataset_files, "No CSV files found in dataset directory"

Dataset dir: D:\UIT\1003_EPA_PROJECT\1003_EPA-Project_UIT\dataset
Found 2 CSV files
 - DAX_40.csv
 - EuroNext_100.csv


In [7]:
import importlib
import AAAI24_GARCH_NN_Reproduction.experiments.run_benchmark as run_benchmark_module

run_benchmark_module = importlib.reload(run_benchmark_module)
run_benchmark = run_benchmark_module.run_benchmark

# Run mode:
# - smoke: quick check (run exactly one dataset from _smoke_dataset)
# - full: full reproduction
RUN_MODE = "full"

SEQ_LEN = 126
EPOCHS = 2 if RUN_MODE == "smoke" else 60
BATCH_SIZE = 64
DEVICE = DEFAULT_DEVICE
NUM_WORKERS = 2 if str(DEVICE).startswith("cuda") else 0
LOG_PROGRESS = True

if RUN_MODE == "smoke":
    smoke_dir = PROJECT_ROOT / "AAAI24_GARCH_NN_Reproduction" / "experiments" / "_smoke_dataset"
    smoke_files = sorted(smoke_dir.glob("*.csv"))
    assert smoke_files, f"No CSV files found in smoke dataset dir: {smoke_dir}"

    # Use only one dataset file in smoke mode.
    DATASET_DIR = smoke_files[0]
    dataset_files = [DATASET_DIR]
else:
    DATASET_DIR = PROJECT_ROOT / "dataset"
    dataset_files = sorted(DATASET_DIR.glob("*.csv"))
    assert dataset_files, f"No CSV files found in dataset dir: {DATASET_DIR}"

if str(DEVICE).startswith("cuda"):
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")

RESULTS_DIR = PROJECT_ROOT / "AAAI24_GARCH_NN_Reproduction" / "experiments" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Output type 1: benchmark metrics per dataset/seed/horizon/model
OUTPUT_CSV = RESULTS_DIR / f"benchmark_results_{RUN_MODE}.csv"

# Output type 2: time-level volatility (mean over seeds)
OUTPUT_MEAN_SEED_CSV = OUTPUT_CSV.with_name(
    OUTPUT_CSV.stem + "_predictions_mean_seed.csv"
 )

print(f"Run mode: {RUN_MODE}")
print(f"Dataset source: {DATASET_DIR}")
print("Datasets to run:")
for p in dataset_files:
    print(" -", p.name)
print(f"Seq len: {SEQ_LEN}")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Device: {DEVICE}")
print(f"Num workers: {NUM_WORKERS}")
print(f"Log progress: {LOG_PROGRESS}")
print(f"Output 1 (metrics): {OUTPUT_CSV}")
print(f"Output 2 (mean-seed volatility): {OUTPUT_MEAN_SEED_CSV}")

Run mode: full
Dataset source: D:\UIT\1003_EPA_PROJECT\1003_EPA-Project_UIT\dataset
Datasets to run:
 - DAX_40.csv
 - EuroNext_100.csv
Seq len: 126
Epochs: 60
Batch size: 64
Device: cuda:0
Num workers: 2
Log progress: True
Output 1 (metrics): D:\UIT\1003_EPA_PROJECT\1003_EPA-Project_UIT\AAAI24_GARCH_NN_Reproduction\experiments\results\benchmark_results_full.csv
Output 2 (mean-seed volatility): D:\UIT\1003_EPA_PROJECT\1003_EPA-Project_UIT\AAAI24_GARCH_NN_Reproduction\experiments\results\benchmark_results_full_predictions_mean_seed.csv


In [8]:
import pandas as pd

print(f"Calling run_benchmark on: {DATASET_DIR}")
print(f"log_progress: {LOG_PROGRESS}")

results_df = run_benchmark(
    dataset_dir=DATASET_DIR,
    seq_len=SEQ_LEN,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    output_csv=OUTPUT_CSV,
    device=DEVICE,
    num_workers=NUM_WORKERS,
    log_progress=LOG_PROGRESS,
)

assert OUTPUT_CSV.exists(), f"Missing output file: {OUTPUT_CSV}"
assert OUTPUT_MEAN_SEED_CSV.exists(), f"Missing output file: {OUTPUT_MEAN_SEED_CSV}"

mean_seed_df = pd.read_csv(OUTPUT_MEAN_SEED_CSV)

print(f"Output 1 saved: {OUTPUT_CSV}")
print(f"Output 2 saved: {OUTPUT_MEAN_SEED_CSV}")
print(f"Output 1 rows: {len(results_df)}")
print(f"Output 2 rows: {len(mean_seed_df)}")

print("\nOutput 1 preview (benchmark metrics):")
display(results_df.head())

print("\nOutput 2 preview (mean-seed volatility):")
display(mean_seed_df.head())

Calling run_benchmark on: D:\UIT\1003_EPA_PROJECT\1003_EPA-Project_UIT\dataset
log_progress: True
[10:11:50] Benchmark started | device=cuda:0 | datasets=2 | seeds=5 | horizons=[1, 3, 5, 10, 21]
[10:11:50] Dataset start: DAX_40.csv
[10:11:50] [DAX_40] Seed 42 started
[10:11:50] [DAX_40][seed=42] Training Autoformer...
[10:12:36] [DAX_40][seed=42] Done Autoformer | epochs=43 | best_val_loss=0.369262 | time=45.6s
[10:12:36] [DAX_40][seed=42] Training Informer...
[10:12:56] [DAX_40][seed=42] Done Informer | epochs=24 | best_val_loss=0.503362 | time=19.7s
[10:12:56] [DAX_40][seed=42] Training Reformer...
[10:13:12] [DAX_40][seed=42] Done Reformer | epochs=27 | best_val_loss=0.347525 | time=16.7s
[10:13:12] [DAX_40][seed=42] Training Transformer...
[10:13:32] [DAX_40][seed=42] Done Transformer | epochs=22 | best_val_loss=0.450054 | time=19.8s
[10:13:32] [DAX_40][seed=42] Training GARCH-LSTM-Hybrid...
[10:20:06] [DAX_40][seed=42] Done GARCH-LSTM-Hybrid | epochs=34 | best_val_loss=0.338358 | 

,Dataset,Seed,Horizon,Model,MAE,MSE,N_eval,Seq_len
0,DAX_40,42,1,GARCH,0.572662,0.516624,456,126
1,DAX_40,42,1,GJR-GARCH,0.561986,0.507343,456,126
2,DAX_40,42,1,FI-GARCH,0.567174,0.506536,456,126
3,DAX_40,42,1,Autoformer,0.515532,0.448187,456,126
4,DAX_40,42,1,Informer,0.696549,0.668220,456,126



Output 2 preview (mean-seed volatility):


,time,dataset,model,horizon,True_Volatility,Pred_Volatility,time_train
0,2024-03-13 00:00:00,DAX_40,Autoformer,1,0.020756,1.211509,42.869573
1,2024-03-14 00:00:00,DAX_40,Autoformer,1,0.107743,1.072861,42.869573
2,2024-03-15 00:00:00,DAX_40,Autoformer,1,0.030038,0.867724,42.869573
3,2024-03-18 00:00:00,DAX_40,Autoformer,1,0.022140,0.779410,42.869573
4,2024-03-19 00:00:00,DAX_40,Autoformer,1,0.305180,0.908617,42.869573


In [9]:
import pandas as pd

# Inspect output type 1 (benchmark metrics CSV)
output1_df = pd.read_csv(OUTPUT_CSV)
output1_summary_df = (
    output1_df.groupby(["Dataset", "Model", "Horizon"], as_index=False)[["MAE", "MSE"]]
    .mean()
    .sort_values(["Dataset", "Model", "Horizon"])
)

print(f"Loaded Output 1: {OUTPUT_CSV}")
print(f"Rows: {len(output1_df)}")
print(f"Columns: {list(output1_df.columns)}")

output1_summary_df.head(30)

Loaded Output 1: D:\UIT\1003_EPA_PROJECT\1003_EPA-Project_UIT\AAAI24_GARCH_NN_Reproduction\experiments\results\benchmark_results_full.csv
Rows: 400
Columns: ['Dataset', 'Seed', 'Horizon', 'Model', 'MAE', 'MSE', 'N_eval', 'Seq_len']


,Dataset,Model,Horizon,MAE,MSE
0,DAX_40,Autoformer,1,0.532798,0.463524
1,DAX_40,Autoformer,3,0.651210,0.834947
2,DAX_40,Autoformer,5,0.757003,1.195455
3,DAX_40,Autoformer,10,0.958148,2.032586
4,DAX_40,Autoformer,21,1.239463,3.566679
5,DAX_40,FI-GARCH,1,0.567174,0.506536
6,DAX_40,FI-GARCH,3,0.710041,0.946035
7,DAX_40,FI-GARCH,5,0.828781,1.343688
8,DAX_40,FI-GARCH,10,1.087083,2.341647
9,DAX_40,FI-GARCH,21,1.333931,3.718293


In [ ]:
import pandas as pd

# Inspect output type 2 (mean-seed volatility CSV)
output2_df = pd.read_csv(OUTPUT_MEAN_SEED_CSV)
required_columns = [
    "time",
    "dataset",
    "model",
    "horizon",
    "True_Volatility",
    "Pred_Volatility",
    "time_train",
]
missing_columns = [c for c in required_columns if c not in output2_df.columns]
assert not missing_columns, f"Missing columns in output2: {missing_columns}"

print(f"Loaded Output 2: {OUTPUT_MEAN_SEED_CSV}")
print(f"Rows: {len(output2_df)}")
print(f"Columns: {list(output2_df.columns)}")

# Quick view for horizon=1 (typical plotting case)
output2_h1_df = output2_df[output2_df["horizon"] == 1].copy()
print(f"Rows with horizon=1: {len(output2_h1_df)}")

output2_h1_df.head(30)